In [ ]:
import os
import json
from langchain import OpenAI, ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Configuration
LLM_API_KEY = "your_openai_api_key_here"  # Replace with your actual OpenAI API key
QUERY_FILE = "queries.json"
LOG_FILE = "conversation_logs.json"

# Pre-loaded expert user data from the table
EXPERT_DATA = {
    "expert_user_1": {"skills": ["Kubernetes", "Java", "Ansible", "Python", "GCP", "MongoDB"], "experience": 7, "location": "Bangalore"},
    "expert_user_2": {"skills": ["Docker", "Jenkins", "Azure", "Kubernetes", "OpenShift"], "experience": 11, "location": "Delhi"},
    "expert_user_3": {"skills": ["GCP", "Ansible", "DevOps"], "experience": 12, "location": "Pune"},
    "expert_user_4": {"skills": ["React", "Python", "Cybersecurity"], "experience": 5, "location": "Remote"},
    "expert_user_5": {"skills": ["Python", "Linux", "Jenkins", "Kubernetes"], "experience": 8, "location": "Delhi"},
    "expert_user_6": {"skills": ["OpenShift", "GCP", "Azure", "Jenkins", "Node.js", "AWS"], "experience": 3, "location": "Bangalore"},
    "expert_user_7": {"skills": ["DevOps", "OpenShift", "Ansible", "Terraform", "Java", "Python"], "experience": 10, "location": "Bangalore"},
    "expert_user_8": {"skills": ["Data Science", "PostgreSQL", "Terraform", "DevOps"], "experience": 14, "location": "Pune"},
    "expert_user_9": {"skills": ["Machine Learning", "Jenkins", "OpenShift"], "experience": 7, "location": "Pune"},
    "expert_user_10": {"skills": ["Docker", "Kubernetes", "Cybersecurity", "DevOps", "Machine Learning", "PostgreSQL"], "experience": 4, "location": "Bangalore"},
    "expert_user_11": {"skills": ["OpenShift", "Jenkins", "MongoDB", "React", "Data Science"], "experience": 4, "location": "Chennai"},
    "expert_user_12": {"skills": ["Ansible", "Python", "Jenkins", "MongoDB"], "experience": 7, "location": "Bangalore"},
    "expert_user_13": {"skills": ["Java", "React", "OpenShift", "Python", "Ansible"], "experience": 2, "location": "Bangalore"},
    "expert_user_14": {"skills": ["Jenkins", "Kubernetes", "OpenShift", "Cybersecurity", "Azure", "Docker"], "experience": 6, "location": "Hyderabad"},
    "expert_user_15": {"skills": ["Azure", "Cybersecurity", "Python", "Machine Learning"], "experience": 8, "location": "Chennai"},
    "expert_user_16": {"skills": ["Machine Learning", "DevOps", "Terraform", "Data Science"], "experience": 13, "location": "Bangalore"},
    "expert_user_17": {"skills": ["Node.js", "Kubernetes", "GCP"], "experience": 9, "location": "Delhi"},
    "expert_user_18": {"skills": ["Azure", "GCP", "Kubernetes", "Docker", "Node.js"], "experience": 7, "location": "Chennai"},
    "expert_user_19": {"skills": ["Linux", "Jenkins", "Java", "React", "DevOps", "Terraform"], "experience": 4, "location": "Pune"},
    "expert_user_20": {"skills": ["React", "Ansible", "MongoDB", "OpenShift", "AWS", "PostgreSQL"], "experience": 14, "location": "Remote"}
}

# Pre-loaded platform information
PLATFORM_INFO = {
    "how_it_works": "The platform connects seekers with experts for various services. Seekers can browse expert profiles, book services, and get support.",
    "booking_process": "To book a service, seekers need to select an expert, choose a service, and confirm the booking through the app.",
    "profile_setup": "Experts can set up their profiles by providing their skills, experience, and location. Verification is done through a review process."
}

# Initialize LLM
llm = OpenAI(api_key=LLM_API_KEY)

# Conversation memory
memory = ConversationBufferMemory()

# Prompt template
prompt_template = PromptTemplate(
    input_variables=["history", "input"],
    template="You are a helpful assistant on a platform connecting seekers and experts. {history}\nUser: {input}\nAssistant:"
)

# Conversation chain
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=prompt_template
)

def load_queries():
    if os.path.exists(QUERY_FILE):
        with open(QUERY_FILE, "r") as f:
            return json.load(f)
    return {}

def save_queries(queries):
    with open(QUERY_FILE, "w") as f:
        json.dump(queries, f)

def log_conversation(role, user_input, response):
    log_entry = {"role": role, "user_input": user_input, "response": response}
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r") as f:
            logs = json.load(f)
    else:
        logs = []
    logs.append(log_entry)
    with open(LOG_FILE, "w") as f:
        json.dump(logs, f)

def search_experts(query, role):
    """Search expert data based on user query."""
    results = []
    query = query.lower()
    for username, info in EXPERT_DATA.items():
        skills = [skill.lower() for skill in info["skills"]]
        location = info["location"].lower()
        experience = info["experience"]
        
        # Example seeker queries
        if role == "seeker":
            if "in" in query and any(loc in query for loc in [location]):
                if any(skill in query for skill in skills):
                    results.append(f"{username}: Skills - {', '.join(info['skills'])}, Experience - {experience} years, Location - {info['location']}")
            elif "over" in query or "more than" in query:
                try:
                    years = int(''.join(filter(str.isdigit, query)))
                    if experience > years:
                        results.append(f"{username}: Skills - {', '.join(info['skills'])}, Experience - {experience} years, Location - {info['location']}")
                except ValueError:
                    pass
            elif any(skill in query for skill in skills):
                results.append(f"{username}: Skills - {', '.join(info['skills'])}, Experience - {experience} years, Location - {info['location']}")
        
        # Example expert queries
        elif role == "expert" and username in query:
            results.append(f"Your profile ({username}): Skills - {', '.join(info['skills'])}, Experience - {experience} years, Location - {info['location']}")
    
    return "\n".join(results) if results else None

def handle_query(role, user_input):
    queries = load_queries()
    
    # Check stored queries first
    if user_input in queries:
        response = queries[user_input]
    else:
        # Check platform info
        for key, value in PLATFORM_INFO.items():
            if key in user_input.lower():
                response = value
                break
        else:
            # Search expert data
            expert_results = search_experts(user_input, role)
            if expert_results:
                response = expert_results
            else:
                # Fallback to LLM
                response = conversation.predict(input=user_input)
        # Save new query
        queries[user_input] = response
        save_queries(queries)
    
    # Log the conversation
    log_conversation(role, user_input, response)
    return response

def main():
    print("Hello! Are you a seeker or an expert?")
    role = input("You: ").strip().lower()
    if role not in ["seeker", "expert"]:
        print("Invalid role. Please specify 'seeker' or 'expert'.")
        return
    print(f"Great! How can I assist you today as a {role}?")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break
        response = handle_query(role, user_input)
        print(f"Assistant: {response}")

if __name__ == "__main__":
    main()